# Example 10.4.2 / 1: Solve a DPWL Relation with Binary Segments

Companion notebook for **10.4.2 Discontinuous Piecewise-Linear (DPWL) Relations** in the AMPL lab Documentation, section "Solve a DPWL Relation with Binary Segments". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a DPWL Relation with Binary Segments

The same relation can be written with one binary variable per segment between consecutive
points. `Seg[s] = 1` selects segment `s`, and only the two points at its ends may have positive
weight. This form runs with any MIP solver, including HiGHS:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set POINTS ordered;  # graph points in order; a jump repeats an x value
param x_value {POINTS};  # x coordinate of each point
param y_value {POINTS};  # y coordinate of each point
param demand >= 0;  # smallest quantity required

set SEGMENTS = {k in POINTS: ord(k) < card(POINTS)};  # segment from k to next(k)

var Lambda {POINTS} >= 0, <= 1;  # convex-combination weights
var Seg {SEGMENTS} binary;  # 1 for the segment in use
var x;  # quantity
var y;  # cost

minimize Cost: y;

subject to Convex_Weights: sum {k in POINTS} Lambda[k] = 1;
subject to Define_X: x = sum {k in POINTS} x_value[k] * Lambda[k];
subject to Define_Y: y = sum {k in POINTS} y_value[k] * Lambda[k];
subject to One_Segment: sum {s in SEGMENTS} Seg[s] = 1;
subject to Adjacent {k in POINTS}:  # a point is used only by a selected neighbor segment
    Lambda[k] <= sum {s in SEGMENTS: s = k or next(s, POINTS) = k} Seg[s];
subject to Need: x >= demand;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["POINTS"] = [1, 2, 3, 4]
ampl.param["x_value"] = {1: 0, 2: 10, 3: 10, 4: 20}
ampl.param["y_value"] = {1: 0, 2: 20, 3: 35, 4: 45}
ampl.param["demand"] = 12

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
print("x =", ampl.var["x"].value(), " y =", ampl.var["y"].value())
print(ampl.var["Lambda"].to_dict())
print(ampl.var["Seg"].to_dict())

solved
x = 12.0  y = 37.0
{1: 0, 2: 0, 3: 0.8, 4: 0.2}
{1: 0, 2: 0, 3: 1}


### Pure AMPL: data.dat and model.run

The demand of 12 lies past the jump at 10, so the cost is read on the piece after the jump,
segment 3 from point 3 `(10, 35)` to point 4 `(20, 45)`: weights 0.8 and 0.2 give
`y = 0.8 * 35 + 0.2 * 45 = 37`. Segment 2, between the two copies of `x = 10`, is the vertical
jump itself. Without the segment rule (only `Convex_Weights`, `Define_X` and `Define_Y`), the
weights could mix points 2 and 4 and report a cost of 25, which lies below the graph.

Pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set POINTS := 1 2 3 4;
param: x_value y_value :=
    1    0    0
    2   10   20
    3   10   35
    4   20   45;
param demand := 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
option solver_msg 0;
solve;
display x, y;
display Lambda, Seg;

x = 12
y = 37

: Lambda Seg    :=
1   0      0
2   0      0
3   0.8    1
4   0.2    .
;

